# Data Quality & Reconciliation

Data quality checks are performed to ensure that the healthcare
data is accurate, complete, consistent, and reliable before it is
used for analytics.

The following validations are implemented:

1. Record count reconciliation
2. Duplicate record detection
3. NULL value validation
4. Referential integrity validation
5. Invalid date detection
6. Financial data validation
7. Bronze-to-Silver reconciliation
8. Gold fact-table reconciliation

These checks help identify data issues before they affect
healthcare analytics and reporting.

## 1. Record Count Reconciliation

Record counts between Bronze and Silver are compared to identify
unexpected data loss during the cleansing and transformation process.

A difference in counts can be expected when duplicate records are
removed during Silver-layer cleansing.

In [0]:
from pyspark.sql import functions as F

reconciliation_tables = [
    "patients",
    "doctors",
    "departments",
    "diagnoses",
    "treatments",
    "medications",
    "insurance_providers",
    "patient_addresses",
    "patient_contacts",
    "patient_insurance",
    "doctor_departments",
    "appointments",
    "appointment_diagnoses",
    "appointment_treatments",
    "prescriptions",
    "prescription_items",
    "lab_tests",
    "lab_test_results",
    "billing",
    "insurance_claims",
    "payments"
]

for table in reconciliation_tables:
    bronze_count = spark.table(f"bronze_{table}").count()
    silver_count = spark.table(f"silver_{table}").count()

    print(
        f"{table}: "
        f"Bronze={bronze_count}, "
        f"Silver={silver_count}, "
        f"Difference={bronze_count - silver_count}"
    )

patients: Bronze=100, Silver=100, Difference=0
doctors: Bronze=1, Silver=1, Difference=0
departments: Bronze=1, Silver=1, Difference=0
diagnoses: Bronze=1, Silver=1, Difference=0
treatments: Bronze=1, Silver=1, Difference=0
medications: Bronze=1, Silver=1, Difference=0
insurance_providers: Bronze=1, Silver=1, Difference=0
patient_addresses: Bronze=120, Silver=120, Difference=0
patient_contacts: Bronze=150, Silver=150, Difference=0
patient_insurance: Bronze=120, Silver=120, Difference=0
doctor_departments: Bronze=1, Silver=1, Difference=0
appointments: Bronze=500, Silver=500, Difference=0
appointment_diagnoses: Bronze=600, Silver=600, Difference=0
appointment_treatments: Bronze=600, Silver=600, Difference=0
prescriptions: Bronze=300, Silver=300, Difference=0
prescription_items: Bronze=600, Silver=600, Difference=0
lab_tests: Bronze=400, Silver=400, Difference=0
lab_test_results: Bronze=400, Silver=400, Difference=0
billing: Bronze=500, Silver=500, Difference=0
insurance_claims: Bronze=2

## 2. Duplicate Record Detection

Duplicate records can cause incorrect counts and inaccurate healthcare
analytics.

The Silver layer removes duplicates during cleansing. This validation
checks whether any duplicate primary-key values remain.

In [0]:
primary_keys = {
    "patients": "patient_id",
    "doctors": "doctor_id",
    "departments": "department_id",
    "diagnoses": "diagnosis_id",
    "treatments": "treatment_id",
    "medications": "medication_id",
    "insurance_providers": "insurance_provider_id",
    "patient_addresses": "address_id",
    "patient_contacts": "contact_id",
    "patient_insurance": "patient_insurance_id",
    "doctor_departments": "doctor_department_id",
    "appointments": "appointment_id",
    "appointment_diagnoses": "appointment_diagnosis_id",
    "appointment_treatments": "appointment_treatment_id",
    "prescriptions": "prescription_id",
    "prescription_items": "prescription_item_id",
    "lab_tests": "lab_test_id",
    "lab_test_results": "lab_result_id",
    "billing": "billing_id",
    "insurance_claims": "claim_id",
    "payments": "payment_id"
}

for table, pk in primary_keys.items():

    df = spark.table(f"silver_{table}")

    duplicate_count = (
        df.groupBy(pk)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{table}: "
        f"{duplicate_count} duplicate {pk} values"
    )

patients: 0 duplicate patient_id values
doctors: 0 duplicate doctor_id values
departments: 0 duplicate department_id values
diagnoses: 0 duplicate diagnosis_id values
treatments: 0 duplicate treatment_id values
medications: 0 duplicate medication_id values
insurance_providers: 0 duplicate insurance_provider_id values
patient_addresses: 0 duplicate address_id values
patient_contacts: 0 duplicate contact_id values
patient_insurance: 0 duplicate patient_insurance_id values
doctor_departments: 0 duplicate doctor_department_id values
appointments: 0 duplicate appointment_id values
appointment_diagnoses: 0 duplicate appointment_diagnosis_id values
appointment_treatments: 0 duplicate appointment_treatment_id values
prescriptions: 0 duplicate prescription_id values
prescription_items: 0 duplicate prescription_item_id values
lab_tests: 0 duplicate lab_test_id values
lab_test_results: 0 duplicate lab_result_id values
billing: 0 duplicate billing_id values
insurance_claims: 0 duplicate claim_id v

## 3. NULL Value Validation

Required fields should not contain NULL values because missing
identifiers or essential healthcare information can affect joins,
reporting, and analytical accuracy.

This validation checks important fields across the Silver layer.

In [0]:
required_columns = {
    "patients": [
        "patient_id",
        "first_name",
        "last_name",
        "date_of_birth",
        "gender"
    ],
    "doctors": [
        "doctor_id",
        "first_name",
        "last_name",
        "specialization"
    ],
    "departments": [
        "department_id",
        "department_name"
    ],
    "appointments": [
        "appointment_id",
        "patient_id",
        "doctor_id",
        "department_id",
        "appointment_date"
    ],
    "billing": [
        "billing_id",
        "appointment_id",
        "patient_id",
        "total_amount",
        "billing_date"
    ],
    "prescriptions": [
        "prescription_id",
        "appointment_id",
        "patient_id"
    ],
    "lab_tests": [
        "lab_test_id",
        "appointment_id",
        "patient_id",
        "test_name"
    ],
    "insurance_claims": [
        "claim_id",
        "billing_id",
        "patient_id",
        "claim_amount"
    ]
}

for table, columns in required_columns.items():

    df = spark.table(f"silver_{table}")

    for column in columns:
        null_count = df.filter(
            F.col(column).isNull()
        ).count()

        print(
            f"{table}.{column}: {null_count} NULL values"
        )

patients.patient_id: 0 NULL values
patients.first_name: 0 NULL values
patients.last_name: 0 NULL values
patients.date_of_birth: 0 NULL values
patients.gender: 0 NULL values
doctors.doctor_id: 0 NULL values
doctors.first_name: 0 NULL values
doctors.last_name: 0 NULL values
doctors.specialization: 0 NULL values
departments.department_id: 0 NULL values
departments.department_name: 0 NULL values
appointments.appointment_id: 0 NULL values
appointments.patient_id: 0 NULL values
appointments.doctor_id: 0 NULL values
appointments.department_id: 0 NULL values
appointments.appointment_date: 0 NULL values
billing.billing_id: 0 NULL values
billing.appointment_id: 0 NULL values
billing.patient_id: 0 NULL values
billing.total_amount: 0 NULL values
billing.billing_date: 0 NULL values
prescriptions.prescription_id: 0 NULL values
prescriptions.appointment_id: 0 NULL values
prescriptions.patient_id: 0 NULL values
lab_tests.lab_test_id: 0 NULL values
lab_tests.appointment_id: 0 NULL values
lab_tests.pati

## 4. Referential Integrity Validation

Referential integrity ensures that foreign-key relationships remain
valid after ingestion and transformation.

The following relationships are validated:

- Appointments → Patients
- Appointments → Doctors
- Appointments → Departments
- Appointment Diagnoses → Appointments
- Appointment Diagnoses → Diagnoses
- Appointment Treatments → Appointments
- Appointment Treatments → Treatments
- Prescriptions → Appointments
- Prescription Items → Prescriptions
- Prescription Items → Medications
- Lab Tests → Appointments
- Lab Test Results → Lab Tests
- Billing → Appointments
- Insurance Claims → Billing
- Payments → Billing

Invalid references are identified using left-anti joins.

In [0]:
# Appointments → Patients / Doctors / Departments

appointments = spark.table("silver_appointments")

patient_ids = spark.table("silver_patients").select("patient_id").distinct()
doctor_ids = spark.table("silver_doctors").select("doctor_id").distinct()
department_ids = spark.table("silver_departments").select("department_id").distinct()

invalid_appointment_patients = appointments.join(
    patient_ids,
    "patient_id",
    "left_anti"
)

invalid_appointment_doctors = appointments.join(
    doctor_ids,
    "doctor_id",
    "left_anti"
)

invalid_appointment_departments = appointments.join(
    department_ids,
    "department_id",
    "left_anti"
)

print(
    "Appointments → Patients:",
    invalid_appointment_patients.count()
)

print(
    "Appointments → Doctors:",
    invalid_appointment_doctors.count()
)

print(
    "Appointments → Departments:",
    invalid_appointment_departments.count()
)

Appointments → Patients: 0
Appointments → Doctors: 0
Appointments → Departments: 0


In [0]:
# Appointment Diagnoses

appointment_diagnoses = spark.table("silver_appointment_diagnoses")
diagnosis_ids = spark.table("silver_diagnoses").select("diagnosis_id").distinct()
appointment_ids = appointments.select("appointment_id").distinct()

print(
    "Appointment Diagnoses → Appointments:",
    appointment_diagnoses.join(
        appointment_ids,
        "appointment_id",
        "left_anti"
    ).count()
)

print(
    "Appointment Diagnoses → Diagnoses:",
    appointment_diagnoses.join(
        diagnosis_ids,
        "diagnosis_id",
        "left_anti"
    ).count()
)


# Appointment Treatments

appointment_treatments = spark.table("silver_appointment_treatments")
treatment_ids = spark.table("silver_treatments").select("treatment_id").distinct()

print(
    "Appointment Treatments → Appointments:",
    appointment_treatments.join(
        appointment_ids,
        "appointment_id",
        "left_anti"
    ).count()
)

print(
    "Appointment Treatments → Treatments:",
    appointment_treatments.join(
        treatment_ids,
        "treatment_id",
        "left_anti"
    ).count()
)

Appointment Diagnoses → Appointments: 0
Appointment Diagnoses → Diagnoses: 0
Appointment Treatments → Appointments: 0
Appointment Treatments → Treatments: 0


In [0]:
# Prescriptions

prescriptions = spark.table("silver_prescriptions")
prescription_items = spark.table("silver_prescription_items")
medication_ids = spark.table("silver_medications").select("medication_id").distinct()
prescription_ids = prescriptions.select("prescription_id").distinct()

print(
    "Prescriptions → Appointments:",
    prescriptions.join(
        appointment_ids,
        "appointment_id",
        "left_anti"
    ).count()
)

print(
    "Prescription Items → Prescriptions:",
    prescription_items.join(
        prescription_ids,
        "prescription_id",
        "left_anti"
    ).count()
)

print(
    "Prescription Items → Medications:",
    prescription_items.join(
        medication_ids,
        "medication_id",
        "left_anti"
    ).count()
)


# Lab Tests

lab_tests = spark.table("silver_lab_tests")
lab_results = spark.table("silver_lab_test_results")
lab_test_ids = lab_tests.select("lab_test_id").distinct()

print(
    "Lab Tests → Appointments:",
    lab_tests.join(
        appointment_ids,
        "appointment_id",
        "left_anti"
    ).count()
)

print(
    "Lab Results → Lab Tests:",
    lab_results.join(
        lab_test_ids,
        "lab_test_id",
        "left_anti"
    ).count()
)


# Billing

billing = spark.table("silver_billing")
billing_ids = billing.select("billing_id").distinct()

print(
    "Billing → Appointments:",
    billing.join(
        appointment_ids,
        "appointment_id",
        "left_anti"
    ).count()
)


# Insurance Claims

claims = spark.table("silver_insurance_claims")

print(
    "Insurance Claims → Billing:",
    claims.join(
        billing_ids,
        "billing_id",
        "left_anti"
    ).count()
)


# Payments

payments = spark.table("silver_payments")

print(
    "Payments → Billing:",
    payments.join(
        billing_ids,
        "billing_id",
        "left_anti"
    ).count()
)

Prescriptions → Appointments: 0
Prescription Items → Prescriptions: 0
Prescription Items → Medications: 0
Lab Tests → Appointments: 0
Lab Results → Lab Tests: 0
Billing → Appointments: 0
Insurance Claims → Billing: 0
Payments → Billing: 0


## 5. Invalid Date Validation

Healthcare dates are validated to identify logically inconsistent
records.

The following rules are checked:

- Patient date of birth must not be in the future.
- Patient date of birth must not be earlier than 1900.
- Appointment dates must be valid.
- Billing dates must not occur before the related appointment.
- Insurance claim dates must not occur before the related billing.
- Payment dates must not occur before the related billing.

In [0]:
from pyspark.sql import functions as F

# 1. Patient DOB validation

patients = spark.table("silver_patients")

invalid_patient_dob = patients.filter(
    (F.col("date_of_birth") > F.current_date()) |
    (F.col("date_of_birth") < F.lit("1900-01-01"))
)

print(
    "Invalid patient DOB records:",
    invalid_patient_dob.count()
)


# 2. Appointment date validation

appointments = spark.table("silver_appointments")

invalid_appointments = appointments.filter(
    F.col("appointment_date").isNull()
)

print(
    "Invalid appointment date records:",
    invalid_appointments.count()
)

Invalid patient DOB records: 0
Invalid appointment date records: 0


In [0]:
# 3. Billing date vs appointment date

billing = spark.table("silver_billing")

appointment_dates = appointments.select(
    "appointment_id",
    "appointment_date"
)

invalid_billing_dates = (
    billing
    .join(
        appointment_dates,
        "appointment_id",
        "left"
    )
    .filter(
        F.col("billing_date") <
        F.to_date("appointment_date")
    )
)

print(
    "Billing before appointment:",
    invalid_billing_dates.count()
)

Billing before appointment: 257


In [0]:
# 4. Claim date vs billing date

claims = spark.table("silver_insurance_claims")

billing_dates = billing.select(
    "billing_id",
    "billing_date"
)

invalid_claim_dates = (
    claims
    .join(
        billing_dates,
        "billing_id",
        "left"
    )
    .filter(
        F.col("claim_date") <
        F.col("billing_date")
    )
)

print(
    "Claims before billing:",
    invalid_claim_dates.count()
)


# 5. Payment date vs billing date

payments = spark.table("silver_payments")

invalid_payment_dates = (
    payments
    .join(
        billing_dates,
        "billing_id",
        "left"
    )
    .filter(
        F.col("payment_date") <
        F.col("billing_date")
    )
)

print(
    "Payments before billing:",
    invalid_payment_dates.count()
)

Claims before billing: 0
Payments before billing: 148


### Date Exception Investigation

Some records were identified where billing or payment dates appear
earlier than the associated appointment or billing dates.

Before treating these records as data-quality failures, the records
are inspected to determine whether the differences represent
legitimate operational timing or inconsistent source data.

The source records are not modified during this investigation.

In [0]:
display(
    invalid_billing_dates
    .select(
        "billing_id",
        "appointment_id",
        "appointment_date",
        "billing_date",
        "total_amount",
        "payment_status"
    )
    .orderBy("appointment_id")
    .limit(20)
)

billing_id,appointment_id,appointment_date,billing_date,total_amount,payment_status
1,1,2025-03-22,2024-04-14,47925.31,Partially Paid
6,6,2026-05-23,2026-02-20,42482.16,Paid
7,7,2025-06-02,2024-03-01,34506.8,Pending
9,9,2025-09-04,2025-09-01,47260.11,Paid
11,11,2025-08-01,2025-05-30,99520.39,Paid
12,12,2024-09-20,2024-08-17,88996.97,Paid
14,14,2024-09-02,2024-05-07,36723.37,Pending
15,15,2024-09-22,2024-06-07,73322.19,Partially Paid
18,18,2026-06-26,2025-04-12,29917.58,Paid
19,19,2025-05-08,2024-01-15,32637.07,Pending


In [0]:
display(
    invalid_payment_dates
    .select(
        "payment_id",
        "billing_id",
        "payment_date",
        "billing_date",
        "payment_amount",
        "payment_status"
    )
    .orderBy("billing_id")
    .limit(20)
)

payment_id,billing_id,payment_date,billing_date,payment_amount,payment_status
2,2,2024-09-15,2026-05-18,9345.56,Successful
4,4,2024-05-19,2024-07-25,42253.89,Successful
5,5,2025-03-26,2026-03-01,37286.7,Successful
6,6,2024-05-27,2026-02-20,25893.61,Successful
10,10,2024-04-08,2024-09-25,15557.77,Successful
11,11,2024-09-17,2025-05-30,595.91,Successful
13,13,2024-04-21,2024-06-11,30323.94,Successful
15,15,2024-02-02,2024-06-07,32805.39,Successful
17,17,2025-07-06,2026-02-18,15977.1,Successful
21,21,2024-09-06,2025-12-19,40690.37,Successful


In [0]:
billing_date_exceptions = (
    invalid_billing_dates
    .withColumn(
        "days_before_appointment",
        F.datediff(
            F.to_date("appointment_date"),
            F.col("billing_date")
        )
    )
)

display(
    billing_date_exceptions
    .select(
        "billing_id",
        "appointment_id",
        "appointment_date",
        "billing_date",
        "days_before_appointment"
    )
    .orderBy(F.desc("days_before_appointment"))
    .limit(20)
)

billing_id,appointment_id,appointment_date,billing_date,days_before_appointment
347,347,2026-06-11,2024-01-07,886
216,216,2026-07-23,2024-03-09,866
280,280,2026-07-05,2024-02-22,864
350,350,2026-07-11,2024-03-21,842
393,393,2026-07-19,2024-04-02,838
62,62,2026-07-27,2024-04-21,827
437,437,2026-06-16,2024-03-26,812
152,152,2026-06-01,2024-03-15,808
119,119,2026-08-07,2024-06-08,790
169,169,2026-06-28,2024-05-14,775


In [0]:
payment_date_exceptions = (
    invalid_payment_dates
    .withColumn(
        "days_before_billing",
        F.datediff(
            F.col("billing_date"),
            F.col("payment_date")
        )
    )
)

display(
    payment_date_exceptions
    .select(
        "payment_id",
        "billing_id",
        "billing_date",
        "payment_date",
        "days_before_billing"
    )
    .orderBy(F.desc("days_before_billing"))
    .limit(20)
)

payment_id,billing_id,billing_date,payment_date,days_before_billing
199,199,2026-08-17,2024-02-01,928
84,84,2026-07-28,2024-01-13,927
137,137,2026-05-20,2024-01-20,851
289,289,2026-04-29,2024-01-20,830
132,132,2026-03-16,2024-01-30,776
283,283,2026-07-10,2024-06-12,758
138,138,2026-07-15,2024-06-21,754
149,149,2026-05-10,2024-05-01,739
66,66,2026-06-18,2024-06-15,733
251,251,2026-05-18,2024-05-20,728


### Date Validation Result

The date-quality checks identified inconsistent billing and payment
dates in the source dataset.

- Patient date-of-birth validation: Passed
- Appointment date validation: Passed
- Insurance claim date validation: Passed
- Billing date validation: Exceptions detected
- Payment date validation: Exceptions detected

Billing records were found with billing dates earlier than their
associated appointment dates. Payment records were also found with
payment dates earlier than their associated billing dates.

These records are retained in the pipeline but flagged as data-quality
exceptions rather than being deleted. This preserves source data
lineage while making the issue visible for downstream monitoring.

In [0]:
print("Date Quality Summary")
print("--------------------")
print("Invalid patient DOB:", invalid_patient_dob.count())
print("Invalid appointment dates:", invalid_appointments.count())
print("Billing before appointment:", invalid_billing_dates.count())
print("Claims before billing:", invalid_claim_dates.count())
print("Payments before billing:", invalid_payment_dates.count())

Date Quality Summary
--------------------
Invalid patient DOB: 0
Invalid appointment dates: 0
Billing before appointment: 257
Claims before billing: 0
Payments before billing: 148


## 6. Financial Data Validation

Healthcare billing records are validated using financial
reconciliation rules.

The following relationships are checked:

1. Gross amount - discount + tax should equal total amount.
2. Insurance amount + patient amount should equal total amount.
3. Financial amounts should not be negative.

These checks help identify incorrect billing calculations and
financial inconsistencies before analytical reporting.

In [0]:
billing = spark.table("silver_billing")

financial_validation = (
    billing
    .withColumn(
        "calculated_total",
        F.col("gross_amount")
        - F.col("discount_amount")
        + F.col("tax_amount")
    )
    .withColumn(
        "amount_difference",
        F.round(
            F.col("total_amount") - F.col("calculated_total"),
            2
        )
    )
    .withColumn(
        "coverage_difference",
        F.round(
            F.col("total_amount")
            - (
                F.col("insurance_amount")
                + F.col("patient_amount")
            ),
            2
        )
    )
)

print(
    "Incorrect billing calculation:",
    financial_validation
    .filter(F.col("amount_difference") != 0)
    .count()
)

print(
    "Insurance + patient amount mismatch:",
    financial_validation
    .filter(F.col("coverage_difference") != 0)
    .count()
)

Incorrect billing calculation: 0
Insurance + patient amount mismatch: 0


In [0]:
negative_amount_records = billing.filter(
    (F.col("gross_amount") < 0) |
    (F.col("discount_amount") < 0) |
    (F.col("tax_amount") < 0) |
    (F.col("total_amount") < 0) |
    (F.col("insurance_amount") < 0) |
    (F.col("patient_amount") < 0)
)

print(
    "Records with negative financial amounts:",
    negative_amount_records.count()
)

Records with negative financial amounts: 0


## 7. Gold Fact Reconciliation

The Gold fact tables are reconciled against their corresponding
Silver-layer source tables.

This ensures that the dimensional transformation does not
unexpectedly lose healthcare events.

The expected relationships are:

- Patient Visits → Appointments
- Treatments → Appointment Treatments
- Prescriptions → Prescription Items
- Lab Tests → Lab Tests
- Billing → Billing
- Insurance Claims → Insurance Claims

In [0]:
gold_reconciliation = {
    "fact_patient_visits": "silver_appointments",
    "fact_treatments": "silver_appointment_treatments",
    "fact_prescriptions": "silver_prescription_items",
    "fact_lab_tests": "silver_lab_tests",
    "fact_billing": "silver_billing",
    "fact_insurance_claims": "silver_insurance_claims"
}

for gold_table, silver_table in gold_reconciliation.items():

    gold_count = spark.table(gold_table).count()
    silver_count = spark.table(silver_table).count()

    print(
        f"{gold_table}: "
        f"Gold={gold_count}, "
        f"Silver={silver_count}, "
        f"Difference={silver_count - gold_count}"
    )

fact_patient_visits: Gold=500, Silver=500, Difference=0
fact_treatments: Gold=600, Silver=600, Difference=0
fact_prescriptions: Gold=600, Silver=600, Difference=0
fact_lab_tests: Gold=400, Silver=400, Difference=0
fact_billing: Gold=500, Silver=500, Difference=0
fact_insurance_claims: Gold=200, Silver=200, Difference=0


## 8. Overall Data Quality Summary

The healthcare data pipeline performs quality validation across
multiple dimensions of data quality.

### Validation Results

| Quality Check | Result |
|---|---|
| Bronze → Silver reconciliation | Passed |
| Duplicate primary-key check | Passed |
| Required-field NULL check | Passed |
| Referential integrity | Passed |
| Patient DOB validation | Passed |
| Appointment date validation | Passed |
| Insurance claim date validation | Passed |
| Billing date validation | Exceptions detected |
| Payment date validation | Exceptions detected |
| Billing calculation | Passed |
| Insurance + patient amount reconciliation | Passed |
| Negative financial amounts | Passed |
| Silver → Gold fact reconciliation | Passed |

Billing and payment date inconsistencies were detected and retained
as source-data quality exceptions rather than deleting the affected
records.

This demonstrates that the pipeline validates data while preserving
source data lineage.

In [0]:
print("========================================")
print("   HEALTHCARE DATA QUALITY SUMMARY")
print("========================================")

print("Bronze → Silver reconciliation       : PASSED")
print("Duplicate primary-key validation     : PASSED")
print("Required NULL validation             : PASSED")
print("Referential integrity                : PASSED")
print("Patient DOB validation               : PASSED")
print("Financial reconciliation             : PASSED")
print("Silver → Gold reconciliation         : PASSED")

print()
print("Date exceptions requiring review:")
print("Billing before appointment           :", invalid_billing_dates.count())
print("Payment before billing               :", invalid_payment_dates.count())

print()
print("Overall pipeline validation completed.")

   HEALTHCARE DATA QUALITY SUMMARY
Bronze → Silver reconciliation       : PASSED
Duplicate primary-key validation     : PASSED
Required NULL validation             : PASSED
Referential integrity                : PASSED
Patient DOB validation               : PASSED
Financial reconciliation             : PASSED
Silver → Gold reconciliation         : PASSED

Date exceptions requiring review:
Billing before appointment           : 257
Payment before billing               : 148

Overall pipeline validation completed.
